# Lab 1: 多模态视觉语言理解 — Qwen3-VL + OpenVINO

Qwen3-VL 是通义千问 VL 系列最新一代的多模态视觉语言大模型。它在文本理解与生成、视觉感知与推理、长上下文处理、空间与视频动态理解等方面实现了全面升级。

**主要特点:**
- 🖼️ 支持图像和视频理解
- 🌐 支持多语言对话
- 🔍 强大的 OCR 能力，支持 32 种语言
- 🎯 精确的空间感知与目标定位

在本实验中，我们将：
1. 从 ModelScope 下载预转换的 OpenVINO 模型
2. 使用 OpenVINO 加载并运行推理
3. 体验交互式 Gradio 多模态对话演示

#### 目录：
- [下载模型](#下载模型)
- [选择推理设备](#选择推理设备)
- [加载模型](#加载模型)
- [运行推理](#运行推理)
- [交互式演示](#交互式演示)

## 下载模型
[返回目录 ⬆️](#目录：)

从 ModelScope 下载已经转换并量化好的 Qwen3-VL-4B OpenVINO INT4 模型。如果模型已存在则跳过下载。

In [ ]:
from pathlib import Path

model_dir = Path("Qwen3-VL-4B-Instruct-int4-ov")

if not model_dir.exists():
    from modelscope import snapshot_download
    snapshot_download("snake7gun/Qwen3-VL-4B-Instruct-int4-ov", local_dir=str(model_dir))
    print(f"模型已下载到: {model_dir}")
else:
    print(f"模型已存在: {model_dir}，跳过下载")

## 选择推理设备
[返回目录 ⬆️](#目录：)

选择用于推理的设备。CPU 兼容性最好，GPU 可在支持的硬件上提供加速。

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="AUTO", exclude=["NPU"])
device

## 加载模型
[返回目录 ⬆️](#目录：)

使用 Optimum Intel 的 `OVModelForVisualCausalLM` 加载 OpenVINO 模型。该接口与 Transformers 的实现兼容，可以无缝集成。

In [ ]:
from optimum.intel.openvino import OVModelForVisualCausalLM

model = OVModelForVisualCausalLM.from_pretrained(model_dir, device=device.value)
print("✅ 模型加载完成")

## 运行推理
[返回目录 ⬆️](#目录：)

我们来用一张示例图片测试模型的多模态理解能力。

In [ ]:
from PIL import Image
from transformers import AutoProcessor, TextStreamer
from pathlib import Path
import requests

min_pixels = 256 * 28 * 28
max_pixels = 1280 * 28 * 28
processor = AutoProcessor.from_pretrained(model_dir, min_pixels=min_pixels, max_pixels=max_pixels)

# 下载示例图片
example_image_url = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
example_image_path = Path("demo.jpeg")

if not example_image_path.exists():
    Image.open(requests.get(example_image_url, stream=True).raw).save(example_image_path)

image = Image.open(example_image_path)
question = "请描述这张图片的内容。"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "file": example_image_path},
            {"type": "text", "text": question},
        ],
    }
]

inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt")

display(image)
print(f"问题: {question}")
print("回答:")

generated_ids = model.generate(**inputs, max_new_tokens=200, streamer=TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True))

## 交互式演示
[返回目录 ⬆️](#目录：)

现在你可以通过 Gradio 界面与模型进行交互式对话。支持上传图片或视频，输入文字提问。

In [ ]:
from gradio_helper import make_demo

demo = make_demo(model, processor, model_name="Qwen3_VL")

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(debug=True, share=True)